In [1]:
# Libraries
import pandas as pd
import json
from pathlib import Path
import time

In [2]:
# Folder Paths
INPUT_FILE = Path(
    r"C:\Users\Visakh H S\Labour_Market_Intelligence\data\processed\master_jobs_industry.csv"
)

OUTPUT_FILE = Path(
    r"C:\Users\Visakh H S\Labour_Market_Intelligence\data\processed\master_jobs_llm.csv"
)

CHECKPOINT_FILE = Path(
    r"C:\Users\Visakh H S\Labour_Market_Intelligence\data\processed\llm_checkpoint.csv"
)

df = pd.read_csv(INPUT_FILE)

print("Total jobs:", len(df))
print("Columns:", df.columns.tolist())

Total jobs: 19207
Columns: ['job_id', 'source_id', 'title', 'company', 'location', 'description', 'salary_min', 'salary_max', 'salary_avg', 'created', 'contract_type', 'contract_time', 'url', 'source', 'title_clean', 'description_clean', 'industry']


In [14]:
# Ollama
import ollama

In [15]:
models = ollama.list()

for model in models["models"]:
    print(model["model"])

qwen2.5:7b


In [16]:
print(models)

models=[Model(model='qwen2.5:7b', modified_at=datetime.datetime(2026, 8, 15, 11, 52, 4, 516642, tzinfo=TzInfo(3600)), digest='845dbda0ea48ed749caafd9e6037047aa19acfcfd82e704d7ca97d631a0b697e', size=4683087332, details=ModelDetails(parent_model='', format='gguf', family='qwen2', families=['qwen2'], parameter_size='7.6B', quantization_level='Q4_K_M'))]


In [17]:
# Sample printing of Ollama
sample = df.iloc[0]

print("TITLE:")
print(sample["title"])

print("\nINDUSTRY:")
print(sample["industry"])

print("\nDESCRIPTION:")
print(sample["description_clean"][:1000])

TITLE:
Dispenser NVQ

INDUSTRY:
Healthcare

DESCRIPTION:
join superdrug pharmacy team whether qualified starting welcome application qualified dispenser pharmacy experience eager begin career trainee pharmacy assistant location southport hour hour per week tuesday friday flexibility salary per hour dispenser per hour depending age trainee pharmacy assistant superd


In [18]:
test_title = df.iloc[0]["title"]
test_description = df.iloc[0]["description_clean"]

print("TITLE:")
print(test_title)

print("\nINDUSTRY:")
print(df.iloc[0]["industry"])

print("\nSending to Qwen 7B...")

TITLE:
Dispenser NVQ

INDUSTRY:
Healthcare

Sending to Qwen 7B...


In [19]:
response = ollama.chat(
    model="qwen2.5:7b",
    messages=[
        {
            "role": "user",
            "content": f"""
Analyse this UK job advertisement and extract the main professional skills.

Job title:
{test_title}

Job description:
{test_description[:2500]}

Return a short list of genuine skills only.
"""
        }
    ]
)

print(response["message"]["content"])

- Pharmacy knowledge
- Customer service
- Dispensing
- Flexibility
- Teamwork


In [20]:
# Function
LLM_OUTPUT = {
    "technical_skills": [],
    "soft_skills": [],
    "tools_and_technologies": [],
    "certifications": [],
    "licenses": [],
    "education_required": "",
    "experience_required": "",
    "languages": []
}


def extract_skills(title, description):

    if pd.isna(title):
        title = ""

    if pd.isna(description):
        description = ""

    title = str(title).strip()
    description = str(description)[:2500]

    prompt = f"""
You are an expert Labour Market Intelligence information extraction system.

Analyse ONE UK job advertisement.

Extract ONLY information explicitly stated in the job title or
job description.

Return ONLY valid JSON.

Do NOT use Markdown.
Do NOT explain anything.
Do NOT invent information.
Do NOT guess.

If information is missing:
- use [] for list fields
- use "" for text fields.

IMPORTANT:
Keep each item in the correct category.

TECHNICAL SKILLS:
Professional or job-specific capabilities.

Examples:
Pharmacy knowledge
Dispensing medication
Financial Analysis
Financial Modelling
Risk Management
Data Analysis
Electrical Installation
Project Management
Forklift Operation
Cybersecurity

SOFT SKILLS:
Interpersonal or behavioural competencies.

Examples:
Communication
Teamwork
Leadership
Customer Service
Problem Solving
Attention to Detail
Time Management
Negotiation
Flexibility

Do NOT classify simple personality adjectives such as:
friendly, caring, passionate, lively, enthusiastic, dedicated
as soft skills unless they clearly describe a workplace competency.

TOOLS AND TECHNOLOGIES:
Specific software, programming languages, databases, platforms,
technical systems, technologies or technical equipment.

Examples:
Python
C#
SQL
JavaScript
Microsoft Excel
AutoCAD
SAP
AWS
Hadoop
Apache Spark
Kafka
Docker
Tableau

Do NOT put professional skills here.

CERTIFICATIONS:
Formal certificates or qualifications.

Examples:
NVQ
ACCA
Care Certificate
Food Hygiene Certificate
First Aid Certificate

LICENSES:
Professional, legal or occupational licences.

Examples:
Driving Licence
SIA Licence
CSCS Card
ECS Card
NMC Registration

EDUCATION:
Explicit educational requirements.

Examples:
GCSEs
A-levels
Bachelor's degree
Master's degree
Degree in Computer Science

EXPERIENCE:
Explicit previous experience requirements.

Examples:
2 years of experience
Previous retail experience
Previous management experience

LANGUAGES:
Languages explicitly required or preferred.

Do NOT assume English simply because this is a UK job.

Return EXACTLY this JSON:

{{
    "technical_skills": [],
    "soft_skills": [],
    "tools_and_technologies": [],
    "certifications": [],
    "licenses": [],
    "education_required": "",
    "experience_required": "",
    "languages": []
}}

JOB TITLE:
{title}

JOB DESCRIPTION:
{description}
"""

    try:

        response = ollama.chat(
            model="qwen2.5:7b",
            messages=[
                {
                    "role": "user",
                    "content": prompt
                }
            ]
        )

        text = response["message"]["content"].strip()

        text = text.replace("```json", "")
        text = text.replace("```", "")
        text = text.strip()

        result = json.loads(text)

        for key in LLM_OUTPUT:

            if key not in result:
                result[key] = LLM_OUTPUT[key]

        return result

    except Exception as e:

        print("LLM extraction error:", e)
        return LLM_OUTPUT.copy()

In [21]:
def clean_llm_result(result):

    invalid_values = {
        "",
        "none",
        "none mentioned",
        "not specified",
        "unknown",
        "n/a",
        "no specific tools mentioned"
    }

    list_fields = [
        "technical_skills",
        "soft_skills",
        "tools_and_technologies",
        "certifications",
        "licenses",
        "languages"
    ]

    for field in list_fields:

        if not isinstance(result.get(field), list):
            result[field] = []

        result[field] = [
            str(x).strip()
            for x in result[field]
            if str(x).strip().lower() not in invalid_values
        ]

    # Move obvious certifications/licences
    # that the LLM accidentally placed in tools
    tools = result["tools_and_technologies"]
    certifications = result["certifications"]
    licenses = result["licenses"]

    new_tools = []

    for item in tools:

        item_lower = item.lower()

        if (
            "certificate" in item_lower
            or "nvq" in item_lower
            or "acca" in item_lower
        ):
            if item not in certifications:
                certifications.append(item)

        elif (
            "driving licence" in item_lower
            or "driving license" in item_lower
            or "sia licence" in item_lower
            or "cscs" in item_lower
            or "ecs card" in item_lower
            or "licence" in item_lower
        ):
            if item not in licenses:
                licenses.append(item)

        else:
            new_tools.append(item)

    result["tools_and_technologies"] = new_tools

    # Remove generic objects from tools
    bad_tools = {
        "vehicle",
        "equipment",
        "computer",
        "office"
    }

    result["tools_and_technologies"] = [
        x for x in result["tools_and_technologies"]
        if x.lower() not in bad_tools
    ]

    # Remove obvious personality-only words
    bad_soft_skills = {
        "friendly",
        "caring",
        "passionate",
        "lively",
        "enthusiastic",
        "dedicated",
        "helpful"
    }

    result["soft_skills"] = [
        x for x in result["soft_skills"]
        if x.lower() not in bad_soft_skills
    ]

    # Remove duplicates
    for field in list_fields:
        result[field] = list(dict.fromkeys(result[field]))

    return result

In [ ]:
# Testing ollama
test_results = []

for i, row in df.head(20).iterrows():

    print(f"Processing {i + 1}/20: {row['title']}")

    result = extract_skills(
        row["title"],
        row["description_clean"]
    )

    result = clean_llm_result(result)

    test_results.append(result)

In [ ]:
test_llm_df = pd.DataFrame(test_results)

print(
    test_llm_df.to_string(index=False)
)

In [ ]:
print("\nTECHNICAL SKILLS")

for i, x in enumerate(test_llm_df["technical_skills"]):
    if x:
        print(i, x)

In [ ]:
print("\nSOFT SKILLS")

for i, x in enumerate(test_llm_df["soft_skills"]):
    if x:
        print(i, x)

In [ ]:
print("\nTOOLS AND TECHNOLOGIES")

for i, x in enumerate(test_llm_df["tools_and_technologies"]):
    if x:
        print(i, x)

In [ ]:
print("\nCERTIFICATIONS")

for i, x in enumerate(test_llm_df["certifications"]):
    if x:
        print(i, x)

In [ ]:
print("\nLICENSES")

for i, x in enumerate(test_llm_df["licenses"]):
    if x:
        print(i, x)

In [22]:
# Full dataset LLM extraction with resumable checkpoint version

SAVE_EVERY = 25

RESULT_COLUMNS = [
    "technical_skills",
    "soft_skills",
    "tools_and_technologies",
    "certifications",
    "licenses",
    "education_required",
    "experience_required",
    "languages"
]


# Load existing checkpoint if available

if CHECKPOINT_FILE.exists():

    checkpoint_df = pd.read_csv(CHECKPOINT_FILE)

    if "_row_index" in checkpoint_df.columns:
        processed_indices = set(
            checkpoint_df["_row_index"].astype(int)
        )
    else:
        processed_indices = set()

    print(
        f"Existing checkpoint found: "
        f"{len(processed_indices):,} jobs already completed."
    )

else:

    checkpoint_df = pd.DataFrame()
    processed_indices = set()

    print("No checkpoint found.")
    print("Starting extraction from job 0.")

 
# Start extraction

total_jobs = len(df)

completed_this_run = 0
failed_this_run = 0

start_time = time.time()


for row_index, row in df.iterrows():

# Skip jobs already processed
    if row_index in processed_indices:
        continue

    print(
        f"\nProcessing job "
        f"{row_index + 1}/{total_jobs}: "
        f"{row['title']}"
    )

    try:

# LLM extraction
        result = extract_skills(
            row["title"],
            row["description_clean"]
        )

# Cleaning
        result = clean_llm_result(result)

# Create output record
        record = {
            "_row_index": row_index,
            "job_id": row["job_id"],
            "title": row["title"],
            "industry": row["industry"],
            "status": "success"
        }

# Add extracted fields
        for column in RESULT_COLUMNS:
            record[column] = result.get(
                column,
                [] if column in [
                    "technical_skills",
                    "soft_skills",
                    "tools_and_technologies",
                    "certifications",
                    "licenses",
                    "languages"
                ]
                else ""
            )

    except Exception as e:

        failed_this_run += 1

        print(
            f"ERROR on job {row_index}: {e}"
        )

        record = {
            "_row_index": row_index,
            "job_id": row["job_id"],
            "title": row["title"],
            "industry": row["industry"],
            "status": "failed"
        }

        for column in RESULT_COLUMNS:
            record[column] = (
                []
                if column in [
                    "technical_skills",
                    "soft_skills",
                    "tools_and_technologies",
                    "certifications",
                    "licenses",
                    "languages"
                ]
                else ""
            )

        record["error"] = str(e)


# Add result to checkpoint

    checkpoint_df = pd.concat(
        [
            checkpoint_df,
            pd.DataFrame([record])
        ],
        ignore_index=True
    )

    processed_indices.add(row_index)

    completed_this_run += 1


# Save checkpoint

    if completed_this_run % SAVE_EVERY == 0:

        checkpoint_df.to_csv(
            CHECKPOINT_FILE,
            index=False
        )

        elapsed = time.time() - start_time

        rate = (
            completed_this_run / elapsed
            if elapsed > 0
            else 0
        )

        remaining = total_jobs - len(processed_indices)

        print("\n----------------------------------------")
        print("CHECKPOINT SAVED")
        print("----------------------------------------")

        print(
            f"Completed this run : "
            f"{completed_this_run:,}"
        )

        print(
            f"Total completed    : "
            f"{len(processed_indices):,}/{total_jobs:,}"
        )

        print(
            f"Remaining          : "
            f"{remaining:,}"
        )

        print(
            f"Processing speed   : "
            f"{rate:.3f} jobs/sec"
        )

        print("----------------------------------------")


# Final checkpoint save

checkpoint_df.to_csv(
    CHECKPOINT_FILE,
    index=False
)

print("\n========================================")
print("EXTRACTION RUN FINISHED")
print("========================================")

print(
    f"Completed this run : "
    f"{completed_this_run:,}"
)

print(
    f"Failed this run    : "
    f"{failed_this_run:,}"
)

print(
    f"Total checkpointed : "
    f"{len(checkpoint_df):,}/{total_jobs:,}"
)

print(
    f"Checkpoint file    : "
    f"{CHECKPOINT_FILE}"
)

Existing checkpoint found: 11,500 jobs already completed.

Processing job 11501/19207: Sales Manager

Processing job 11502/19207: Early Years Practitioner

Processing job 11503/19207: Surfacing Operative

Processing job 11504/19207: Customer Sales Assistant

Processing job 11505/19207: Class 2 HIAB Driver

Processing job 11506/19207: Class 1 Driver

Processing job 11507/19207: Class 2 Hiab Driver

Processing job 11508/19207: Class 1 Driver

Processing job 11509/19207: CNC Turner Programmer

Processing job 11510/19207: Class 2 Driver

Processing job 11511/19207: Blinds and Curtains Installer

Processing job 11512/19207: HGV Class 2 Driver

Processing job 11513/19207: Blinds and Curtains Installer

Processing job 11514/19207: HGV Class 1 Driver

Processing job 11515/19207: Sales and Lettings Negotiator

Processing job 11516/19207: Health & Social Care Assessor

Processing job 11517/19207: SEMH Teaching Assistant

Processing job 11518/19207: Experienced SEND TA

Processing job 11519/19207

In [25]:
# Create final LLM dataset

checkpoint_df = pd.read_csv(CHECKPOINT_FILE)

# Remove accidental duplicates
checkpoint_df = checkpoint_df.drop_duplicates(
    subset=["_row_index"],
    keep="last"
)

# Restore original order
checkpoint_df = checkpoint_df.sort_values(
    "_row_index"
)

# Save final file
checkpoint_df.to_csv(
    OUTPUT_FILE,
    index=False
)

print("========================================")
print("FINAL LLM DATASET CREATED")
print("========================================")

print("Rows:", len(checkpoint_df))
print("Output:", OUTPUT_FILE)

print("\nStatus:")
print(
    checkpoint_df["status"]
    .value_counts(dropna=False)
)

FINAL LLM DATASET CREATED
Rows: 19207
Output: master_jobs_llm.csv

Status:
status
success    19207
Name: count, dtype: int64
